# Task 1.2 — Key Assumptions
**Paper**: *Efficient Online Learning for Large-Scale Sparse Kernel Logistic Regression* (AAAI 2012)

## Assumption 1: Bounded Kernel — $\kappa(x, x) \le 1$

**Assumption:** The kernel function satisfies $\kappa(x, x) \le 1$ for every input $x \in \mathbb{R}^d$. This means the feature map induced by the kernel maps every data point onto or inside the unit sphere in the RKHS.

**Why the method needs it:** The bound $\kappa(x,x) \le 1$ directly controls the magnitude of the functional gradient $\nabla_f \ell(y_t f_t(x_t)) = y_t \ell'(y_t f_t(x_t))\,\kappa(x_t, \cdot)$. Both Theorem 1 and Theorem 2 rely on this bound when bounding the one-step progress of the online algorithm; without it, the step-size $\eta$ cannot be set to $R/\sqrt{T}$ and the $O(1/\sqrt{T})$ convergence rate falls apart.

**Violation scenario:** If you use a polynomial kernel $\kappa(x_i, x_j) = (x_i^\top x_j + 1)^d$ on unnormalised data with large feature magnitudes, then $\kappa(x,x)$ can grow unboundedly. In such a setting the gradient norms blow up and the theory gives a vacuous bound, while in practice the algorithm may diverge or require a prohibitively small step size.

**Paper reference:** Stated explicitly in the "Online Learning for KLR" section, right after Eq. (2): "Throughout the paper, we assume $\kappa(x, x) \le 1$ for any $x \in \mathbb{R}^d$."

## Assumption 2: The Optimal Classifier Lies Within the Norm Ball $\Omega = \{f : \|f\|_{\mathcal{H}} \le R\}$

**Assumption:** There exists a value $R$ such that the best-in-class kernel classifier $f^*$ satisfies $\|f^*\|_{\mathcal{H}_\kappa} \le R$. The algorithm never searches outside this ball.

**Why the method needs it:** The convergence guarantees (Theorems 1–4) all measure excess risk relative to $f^* = \arg\min_{f \in \Omega} \mathcal{L}(f)$. If the true minimiser over the full RKHS has a norm larger than $R$, the projection step $\pi_\Omega$ clips the classifier every round and the algorithm can only converge to the best function *inside* the ball, not the global optimum. The value of $R$ therefore controls an implicit bias–variance trade-off.

**Violation scenario:** In a problem where the classes are only separable by a very complex decision boundary (e.g., many overlapping clusters requiring a large-norm function in the RKHS), setting $R$ too small would force the classifier to underfit. The paper selects $R$ via cross-validation from $\{1, 10, \dots, 10^5\}$, but if the true $R$ falls outside this grid the method will silently underperform.

**Paper reference:** Equation (2) defines $\Omega = \{f \in \mathcal{H}_\kappa : \|f\|_{\mathcal{H}_\kappa} \le R\}$; Algorithm 1 line 7 implements the projection; Theorems 1–2 bound the excess risk relative to $f^* \in \Omega$.

## Assumption 3: The Auxiliary Function Dominates the Logit Loss ($h(z) \ge \ell(z)$ everywhere)

**Assumption:** For the Auxiliary algorithm (Algorithm 3), the user-chosen auxiliary function $h(z)$ must satisfy two conditions: (i) $h(z)$ is convex and (ii) $h(z) \ge \ell(z)$ for all $z$. This makes the sampling probability $p_t = \ell(y_t f_t(x_t)) / h(y_t f_t(x_t))$ a valid probability in $[0, 1]$.

**Why the method needs it:** The stochastic conservative update rule multiplies the loss by the Bernoulli indicator $Z_t$. When we take the expectation over $Z_t$, we need $\mathbb{E}[Z_t \cdot h(\cdot)] = p_t \cdot h(\cdot) = \ell(\cdot)$ so that the expected update is an unbiased surrogate for the original logit gradient. If $h(z) < \ell(z)$ for some $z$, then $p_t > 1$, which is not a valid probability, and the entire proof of Theorem 4 breaks down.

**Violation scenario:** Suppose someone accidentally sets $\gamma < 1$ in $h(z) = \ln(\gamma + e^{-z})$. Then for large positive $z$, $h(z) \approx \ln \gamma < 0$ while $\ell(z) > 0$, violating the dominance condition. The algorithm would compute $p_t > 1$ or even negative, leading to nonsensical Bernoulli draws and erratic classifier updates.

**Paper reference:** The two conditions on $h(z)$ are stated right before Theorem 4 in the "Auxiliary function based approach" subsection. Each of the three example auxiliary functions listed below Theorem 4 is verified to satisfy $h(z) \ge \ell(z)$.

## Assumption 4: The Step Size Satisfies $\eta < 2$ (for the Margin Algorithm)

**Assumption:** The learning rate $\eta$ must satisfy $\eta < 2$ for the Margin algorithm's sampling probability $p_t$ (Eq. 5) to lie in $[0, 1]$.

**Why the method needs it:** The Margin sampling probability is $p_t = (2 - \eta) / (2 - \eta + \eta \cdot \sigma(y_t f_t(x_t)))$. If $\eta \ge 2$, the numerator $(2 - \eta)$ becomes non-positive, making $p_t \le 0$ — the Bernoulli gate never fires and no updates occur. The step size $\eta = R/\sqrt{T}$ from Theorem 1 naturally satisfies this for large $T$, but for small datasets or large $R$ the constraint can bind.

**Violation scenario:** On a very small dataset (e.g., $T = 25$ samples) with an aggressive $R = 10$, $\eta = 10/5 = 2.0$, hitting the boundary exactly. The sampling probability degenerates and the Margin algorithm cannot make meaningful conservative updates.

**Paper reference:** The constraint $\eta < 2$ appears explicitly in Theorem 3 (conditions for the Margin algorithm's convergence bound) and is implicit in the denominator of Eq. (5).